In [ ]:
import pandas as pd
from transformers import pipeline
from tqdm import tqdm

from config import INPUT_CSV_PATH, LABELED_CSV_PATH, TEXT_COLUMN, MODEL_NAME


def load_comments(path):
    return pd.read_csv(path)


def create_sentiment_classifier(model_name):
    return pipeline(
        task="sentiment-analysis",
        model=model_name,
        tokenizer=model_name
    )


def predict_sentiment(classifier, text):
    result = classifier(str(text)[:512])[0]
    return result["label"].lower(), result["score"]


def label_comments(df, classifier, text_column):
    sentiments = []
    scores = []

    for text in tqdm(df[text_column], desc="Labeling comments"):
        try:
            sentiment, score = predict_sentiment(classifier, text)
        except Exception:
            sentiment, score = "error", 0.0

        sentiments.append(sentiment)
        scores.append(score)

    df["sentiment"] = sentiments
    df["sentiment_score"] = scores

    return df


df = load_comments(INPUT_CSV_PATH)
classifier = create_sentiment_classifier(MODEL_NAME)

labeled_df = label_comments(df, classifier, TEXT_COLUMN)
labeled_df.to_csv(LABELED_CSV_PATH, index=False)